# Policy change assessment

For each policy change $(p, p')$ we take the projects governed by the old version
(they started before the enactment date), check every trace against the old constraint $p$
and the new constraint $p'$, and count the traces per pair of compliance results.

A compliance result is `vac` (the constraint never activates), `ful` (activated, all activations fulfilled)
or `vio` (at least one activation violated). A missing constraint (addition or deletion) counts as `vac`.

In [1]:
import contextlib
import io
import re
import warnings

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):  # hide the PM4Py banner and import warnings
    import pandas as pd
    import pm4py
    from Declare4Py.D4PyEventLog import D4PyEventLog
    from Declare4Py.ProcessModels.DeclareModel import DeclareModel
    from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
warnings.filterwarnings("ignore")

## 1. Load the log

`boston_permitting_clean.xes` is produced from `boston_permitting.xes` by running `1_infer_project_type.py` and `2_clean_event_log.py` in this order.

- The gross floor area is copied to every event as `gfa`, so that MP-Declare conditions can read it
  on the activating event.
- `large` is 1 for Large Projects and 0 otherwise, so the LOI condition "project type is Large Project"
  can be written as the numeric condition `A.large > 0`.

In [2]:
LOG_PATH = "boston_permitting_clean.xes"

with contextlib.redirect_stderr(io.StringIO()):
    df = pm4py.convert_to_dataframe(pm4py.read_xes(LOG_PATH))

df["gfa"] = df["case:gross_floor_area_sqft"]
# numeric flag for project_type == "Large Project", so the condition can be written as A.large > 0
df["large"] = (df["case:project_type"] == "Large Project").astype(int)

case_start = df.groupby("case:concept:name")["time:timestamp"].min()
print("Projects:", len(case_start))

Projects: 938


## 2. Policy changes

Each change has an old constraint `p` and a new constraint `p'` (`None` if it does not exist),
and the enactment date that separates projects governed by the old version from the new one.

In [3]:
PDA_OLD = "Response[Draft Planned Development Area PDA Development Plan, BPDA Board Approval] | | |60,36500,d"
PDA_NEW = "Response[Draft Planned Development Area PDA Development Plan, BPDA Board Approval] | | |60,90,d"

# paper: Precedence(LOI, PNF) | A.type = Large Project | |
LOI = "Precedence[Letter of Intent LOI, Project Notification Form PNF] |A.large > 0 | |"

BCDC_100K = "Responded Existence[Project Notification Form PNF, Boston Civic Design Commission] |A.gfa > 100000 | |"
BCDC_200K = "Responded Existence[Project Notification Form PNF, Boston Civic Design Commission] |A.gfa > 200000 | |"

CHANGES = pd.DataFrame([
    {"Change": "1 PDA decision window",       "Type": "modification", "p": PDA_OLD,   "p'": PDA_NEW,   "Enacted": "2021-01-14"},
    {"Change": "2 LOI before PNF",            "Type": "addition",     "p": None,      "p'": LOI,       "Enacted": "2025-08-18"},
    {"Change": "3 BCDC threshold 100k->200k", "Type": "modification", "p": BCDC_100K, "p'": BCDC_200K, "Enacted": "2025-08-18"},
]).set_index("Change")
CHANGES

,Type,p,p',Enacted
Change,,,,
1 PDA decision window,modification,Response[Draft Planned Development Area PDA De...,Response[Draft Planned Development Area PDA De...,2021-01-14
2 LOI before PNF,addition,NaN,"Precedence[Letter of Intent LOI, Project Notif...",2025-08-18
3 BCDC threshold 100k->200k,modification,Responded Existence[Project Notification Form ...,Responded Existence[Project Notification Form ...,2025-08-18


## 3. Check all constraints with Declare4Py

All distinct constraints go into one Declare model and are checked in a single run.
`num_activations == 0` gives `vac`; otherwise the state gives `ful` or `vio`.

In [4]:
constraints = list(dict.fromkeys(c for c in pd.concat([CHANGES["p"], CHANGES["p'"]]).dropna()))
activities = list(dict.fromkeys(a.strip() for c in constraints for a in re.search(r"\[(.*?)\]", c).group(1).split(",")))
model = DeclareModel().parse_from_string("\n".join([f"activity {a}" for a in activities] + constraints) + "\n")

log = pm4py.convert_to_event_log(df)
case_ids = [trace.attributes["concept:name"] for trace in log]
results = MPDeclareAnalyzer(log=D4PyEventLog(log=log), declare_model=model, consider_vacuity=True).run()

state = results.get_metric("state").set_axis(case_ids).set_axis(constraints, axis=1)
activations = results.get_metric("num_activations").set_axis(case_ids).set_axis(constraints, axis=1)

verdicts = state.replace({1: "ful", 0: "vio"}).mask(activations == 0, "vac")
verdicts.apply(pd.Series.value_counts).fillna(0).astype(int).set_axis([f"constraint {i+1}" for i in range(len(constraints))], axis=1)

,constraint 1,constraint 2,constraint 3,constraint 4,constraint 5
ful,32,139,9,370,90
vac,895,658,895,534,763
vio,11,141,34,34,85


## 4. Policy change assessments

Rows: compliance result under the old constraint `p`. Columns: compliance result under the new constraint `p'`.
Only projects that started before the enactment date are counted.

| | `vac` | `ful` | `vio` |
|---|---|---|---|
| **`vac`** | not affected | newly bound, compliant | newly bound, violated |
| **`ful`** | released, compliant | unchanged | newly violated |
| **`vio`** | released, violated | newly allowed | unchanged |

In [5]:
VERDICTS = ["vac", "ful", "vio"]

def verdict_pairs(change):
    before = case_start.index[case_start < pd.Timestamp(change["Enacted"], tz="UTC")]
    old = verdicts.loc[before, change["p"]] if pd.notna(change["p"]) else pd.Series("vac", index=before)
    new = verdicts.loc[before, change["p'"]] if pd.notna(change["p'"]) else pd.Series("vac", index=before)
    return pd.crosstab(
        pd.Categorical(old, VERDICTS), pd.Categorical(new, VERDICTS),
        rownames=["p"], colnames=["p'"], dropna=False,
    )

tables = {name: verdict_pairs(change) for name, change in CHANGES.iterrows()}
for name, table in tables.items():
    print(f"{name} ({CHANGES.loc[name, 'Type']})")
    display(table)

1 PDA decision window (modification)


p',vac,ful,vio
p,,,
vac,560,0,0
ful,0,4,14
vio,0,0,8


2 LOI before PNF (addition)


p',vac,ful,vio
p,,,
vac,490,356,31
ful,0,0,0
vio,0,0,0


3 BCDC threshold 100k->200k (modification)


p',vac,ful,vio
p,,,
vac,608,0,0
ful,48,90,0
vio,49,0,82


## 5. Summary

- **Compliant %**: of the projects newly bound by `p'`, the share that already complies
  (high = the change codifies existing practice, low = practice-changing).
- **Newly violated**: projects that complied with `p` but would violate `p'`.
- **Released compliant %**: of the projects released from `p`, the share that complied with it.
  For an obligation this is behavior that is no longer required.
- **Newly allowed**: projects that violated `p` but would comply with `p'`.

In [6]:
def summarize(t):
    bound = t.loc["vac", "ful"] + t.loc["vac", "vio"]
    released = t.loc["ful", "vac"] + t.loc["vio", "vac"]
    return pd.Series({
        "Before": t.values.sum(),
        "Not affected": t.loc["vac", "vac"] + t.loc["ful", "ful"] + t.loc["vio", "vio"],
        "Newly bound": bound,
        "Compliant %": t.loc["vac", "ful"] / bound if bound else None,
        "Newly violated": t.loc["ful", "vio"],
        "Released": released,
        "Released compliant %": t.loc["ful", "vac"] / released if released else None,
        "Newly allowed": t.loc["vio", "ful"],
    })

summary = CHANGES[["Type"]].join(pd.DataFrame({name: summarize(t) for name, t in tables.items()}).T)
summary.style.format({"Compliant %": "{:.0%}", "Released compliant %": "{:.0%}"}, na_rep="—", precision=0)

,Type,Before,Not affected,Newly bound,Compliant %,Newly violated,Released,Released compliant %,Newly allowed
Change,,,,,,,,,
1 PDA decision window,modification,586,572,0,—,14,0,—,0
2 LOI before PNF,addition,877,490,387,92%,0,0,—,0
3 BCDC threshold 100k->200k,modification,877,780,0,—,0,97,49%,0
